## 1. Importação de Bibliotecas e Conexão com MongoDB

In [27]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pymongo import MongoClient
from dotenv import load_dotenv

# Configurações visuais
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Carregar variáveis de ambiente
load_dotenv()

# Conexão MongoDB
MONGO_URI = os.getenv("MONGODB_URI")
mongo_client = MongoClient(MONGO_URI)
db = mongo_client["experiments_db"]
k_alpha_collection = db["refine_judge_similarity_k_alpha"]

# Métricas avaliadas
METRICS = ["coherence", "specificity", "informativeness", "relevance", "Understandability"]
METRIC_LABELS = {
    "coherence": "Coerência",
    "specificity": "Especificidade",
    "informativeness": "Informatividade",
    "relevance": "Relevância",
    "Understandability": "Compreensibilidade"
}

print("✅ Bibliotecas importadas e conexão MongoDB estabelecida!")

✅ Bibliotecas importadas e conexão MongoDB estabelecida!


## 2. Carregamento e Processamento dos Dados

In [28]:
# Carregar todos os documentos da collection
documents = list(k_alpha_collection.find())

print(f"📊 Total de experimentos encontrados: {len(documents)}\n")

# Criar DataFrames para análise
data_rows = []

for doc in documents:
    experiment_name = doc["experiment_name"]
    
    # Dados STI - Krippendorff Alpha médio
    data_rows.append({
        "experiment": experiment_name,
        "metric": "Overall Alpha",
        "approach": "STI",
        "krippendorff_alpha": doc.get("sti_k_alpha", 0),
        "exact_match_percentage": doc.get("sti_exact_match_percentage", 0)
    })
    
    # Dados STI - Alpha por métrica
    if "sti_k_alpha_by_metric" in doc:
        for metric, alpha_value in doc["sti_k_alpha_by_metric"].items():
            # Pegar exact match percentage por métrica se disponível
            exact_match_pct = None
            if "sti_exact_match_percentage_by_metric" in doc:
                exact_match_pct = doc["sti_exact_match_percentage_by_metric"].get(metric)
            
            data_rows.append({
                "experiment": experiment_name,
                "metric": metric,
                "approach": "STI",
                "krippendorff_alpha": alpha_value,
                "exact_match_percentage": exact_match_pct
            })
    
    # Dados MTI - Krippendorff Alpha médio
    data_rows.append({
        "experiment": experiment_name,
        "metric": "Overall Alpha",
        "approach": "MTI",
        "krippendorff_alpha": doc.get("mti_k_alpha", 0),
        "exact_match_percentage": doc.get("mti_exact_match_percentage", 0)
    })
    
    # Dados MTI - Alpha por métrica
    if "mti_k_alpha_by_metric" in doc:
        for metric, alpha_value in doc["mti_k_alpha_by_metric"].items():
            # Pegar exact match percentage por métrica se disponível
            exact_match_pct = None
            if "mti_exact_match_percentage_by_metric" in doc:
                exact_match_pct = doc["mti_exact_match_percentage_by_metric"].get(metric)
            
            data_rows.append({
                "experiment": experiment_name,
                "metric": metric,
                "approach": "MTI",
                "krippendorff_alpha": alpha_value,
                "exact_match_percentage": exact_match_pct
            })

# Criar DataFrame
df = pd.DataFrame(data_rows)

print("✅ Dados processados com sucesso!")
print(f"\n📋 Experimentos encontrados:")
for exp in df["experiment"].unique():
    print(f"   • {exp}")

df.head(20)

📊 Total de experimentos encontrados: 6

✅ Dados processados com sucesso!

📋 Experimentos encontrados:
   • V4
   • GT
   • V5
   • V1
   • V2
   • V3


,experiment,metric,approach,krippendorff_alpha,exact_match_percentage
0,V4,Overall Alpha,STI,0.131,67.917
1,V4,coherence,STI,0.200,63.750
2,V4,specificity,STI,-0.020,47.917
3,V4,informativeness,STI,0.192,62.083
4,V4,relevance,STI,0.277,97.917
5,V4,Understandability,STI,0.005,67.917
6,V4,Overall Alpha,MTI,0.185,66.750
7,V4,coherence,MTI,0.284,64.167
8,V4,specificity,MTI,-0.034,42.500
9,V4,informativeness,MTI,0.259,62.917


## 3. Visualização por Experimento - Krippendorff's Alpha Médio

In [29]:
# Filtrar dados do Overall Alpha
df_overall = df[df["metric"] == "Overall Alpha"].copy()

# Criar gráfico de barras agrupadas para cada experimento
experiments = df_overall["experiment"].unique()

for exp in experiments:
    df_exp = df_overall[df_overall["experiment"] == exp]
    
    fig = go.Figure()
    
    # MTI
    df_mti = df_exp[df_exp["approach"] == "MTI"]
    if not df_mti.empty:
        fig.add_trace(go.Bar(
            name="MTI",
            x=["Krippendorff's Alpha"],
            y=df_mti["krippendorff_alpha"],
            marker_color='rgb(55, 83, 109)',
            text=[f'{val:.2f}' for val in df_mti["krippendorff_alpha"]],
            textposition='outside',
        ))
    
    # STI
    df_sti = df_exp[df_exp["approach"] == "STI"]
    if not df_sti.empty:
        fig.add_trace(go.Bar(
            name="STI",
            x=["Krippendorff's Alpha"],
            y=df_sti["krippendorff_alpha"],
            marker_color='rgb(26, 118, 255)',
            text=[f'{val:.2f}' for val in df_sti["krippendorff_alpha"]],
            textposition='outside',
        ))
    
    # Adicionar linhas de referência para interpretação
    fig.add_hline(y=0.80, line_dash="dash", line_color="green", 
                  annotation_text="Confiável (α ≥ 0.80)", annotation_position="right")
    fig.add_hline(y=0.67, line_dash="dash", line_color="orange",
                  annotation_text="Tentativo (α ≥ 0.67)", annotation_position="right")
    
    fig.update_layout(
        title=f"Krippendorff's Alpha Médio: MTI vs STI<br><sub>Experimento: {exp}</sub>",
        xaxis_title="Métrica",
        yaxis_title="Krippendorff's Alpha",
        barmode='group',
        height=500,
        yaxis=dict(range=[-0.1, 1.1]),
        template="plotly_white"
    )
    
    fig.show()

## 4. Visualização por Experimento - Porcentagem de Respostas Exatas

In [30]:
# Criar gráfico de barras para porcentagem de exact match
for exp in experiments:
    df_exp = df_overall[df_overall["experiment"] == exp]
    
    fig = go.Figure()
    
    # MTI
    df_mti = df_exp[df_exp["approach"] == "MTI"]
    if not df_mti.empty:
        fig.add_trace(go.Bar(
            name="MTI",
            x=["Exact Match %"],
            y=df_mti["exact_match_percentage"],
            marker_color='rgb(55, 83, 109)',
            text=[f'{val:.2f}%' for val in df_mti["exact_match_percentage"]],
            textposition='outside',
        ))
    
    # STI
    df_sti = df_exp[df_exp["approach"] == "STI"]
    if not df_sti.empty:
        fig.add_trace(go.Bar(
            name="STI",
            x=["Exact Match %"],
            y=df_sti["exact_match_percentage"],
            marker_color='rgb(26, 118, 255)',
            text=[f'{val:.2f}%' for val in df_sti["exact_match_percentage"]],
            textposition='outside',
        ))
    
    fig.update_layout(
        title=f"Porcentagem de Respostas Exatas: MTI vs STI<br><sub>Experimento: {exp}</sub>",
        xaxis_title="Métrica",
        yaxis_title="Porcentagem (%)",
        barmode='group',
        height=500,
        yaxis=dict(range=[0, 110]),
        template="plotly_white"
    )
    
    fig.show()

## 7. Gráfico Consolidado - Todos os Experimentos

## 8. Tabela Consolidada Final

In [31]:
# Criar tabela consolidada com todos os experimentos
consolidated_rows = []

for exp in experiments:
    df_exp = df[df["experiment"] == exp]
    
    # Overall Alpha
    df_overall_exp = df_exp[df_exp["metric"] == "Overall Alpha"]
    mti_overall = df_overall_exp[df_overall_exp["approach"] == "MTI"]
    sti_overall = df_overall_exp[df_overall_exp["approach"] == "STI"]
    
    if not mti_overall.empty and not sti_overall.empty:
        mti_alpha = mti_overall["krippendorff_alpha"].values[0]
        sti_alpha = sti_overall["krippendorff_alpha"].values[0]
        mti_exact = mti_overall["exact_match_percentage"].values[0]
        sti_exact = sti_overall["exact_match_percentage"].values[0]
        
        consolidated_rows.append({
            "Experimento": exp,
            "MTI Alpha": f"{mti_alpha:.2f}",
            "STI Alpha": f"{sti_alpha:.2f}",
            "Δ Alpha": f"{mti_alpha - sti_alpha:+.2f}",
            "MTI Exact %": f"{mti_exact:.2f}%",
            "STI Exact %": f"{sti_exact:.2f}%",
            "Δ Exact %": f"{mti_exact - sti_exact:+.2f}%",
            "Vencedor Alpha": "MTI" if mti_alpha > sti_alpha else "STI" if sti_alpha > mti_alpha else "TIE",
            "Vencedor Exact": "MTI" if mti_exact > sti_exact else "STI" if sti_exact > mti_exact else "TIE"
        })

df_consolidated = pd.DataFrame(consolidated_rows)

print("\n" + "="*150)
print("📊 TABELA CONSOLIDADA FINAL - TODOS OS EXPERIMENTOS")
print("="*150)
display(df_consolidated)


📊 TABELA CONSOLIDADA FINAL - TODOS OS EXPERIMENTOS


,Experimento,MTI Alpha,STI Alpha,Δ Alpha,MTI Exact %,STI Exact %,Δ Exact %,Vencedor Alpha,Vencedor Exact
0,V4,0.18,0.13,+0.05,66.75%,67.92%,-1.17%,MTI,STI
1,GT,0.36,0.39,-0.03,77.17%,80.08%,-2.92%,STI,STI
2,V5,0.28,0.14,+0.14,68.50%,68.33%,+0.17%,MTI,MTI
3,V1,-0.29,-0.36,+0.06,32.92%,29.17%,+3.75%,MTI,MTI
4,V2,0.20,0.20,+0.00,68.75%,70.58%,-1.83%,MTI,STI
5,V3,0.19,0.05,+0.14,67.42%,66.00%,+1.42%,MTI,MTI


## 9. Análise de Dominância Global

In [ ]:
# Análise de quantos experimentos cada abordagem venceu
print("\n" + "="*100)
print("🏆 ANÁLISE DE DOMINÂNCIA GLOBAL")
print("="*100)

# Vitórias por Alpha
mti_wins_alpha = (df_diff_overall["diff_alpha"] > 0).sum()
sti_wins_alpha = (df_diff_overall["diff_alpha"] < 0).sum()
ties_alpha = (df_diff_overall["diff_alpha"] == 0).sum()

print("\n📈 Krippendorff's Alpha:")
print(f"  • Vitórias MTI: {mti_wins_alpha}")
print(f"  • Vitórias STI: {sti_wins_alpha}")
print(f"  • Empates: {ties_alpha}")
print(f"  • Abordagem dominante: {'MTI' if mti_wins_alpha > sti_wins_alpha else 'STI' if sti_wins_alpha > mti_wins_alpha else 'BALANCED'}")

# Vitórias por Exact Match
mti_wins_exact = (df_diff_overall["diff_exact"] > 0).sum()
sti_wins_exact = (df_diff_overall["diff_exact"] < 0).sum()
ties_exact = (df_diff_overall["diff_exact"] == 0).sum()

print("\n✅ Exact Match %:")
print(f"  • Vitórias MTI: {mti_wins_exact}")
print(f"  • Vitórias STI: {sti_wins_exact}")
print(f"  • Empates: {ties_exact}")
print(f"  • Abordagem dominante: {'MTI' if mti_wins_exact > sti_wins_exact else 'STI' if sti_wins_exact > mti_wins_exact else 'BALANCED'}")

# Médias gerais
avg_mti_alpha = df_overall[df_overall["approach"] == "MTI"]["krippendorff_alpha"].mean()
avg_sti_alpha = df_overall[df_overall["approach"] == "STI"]["krippendorff_alpha"].mean()
avg_mti_exact = df_overall[df_overall["approach"] == "MTI"]["exact_match_percentage"].mean()
avg_sti_exact = df_overall[df_overall["approach"] == "STI"]["exact_match_percentage"].mean()

print("\n📊 Médias Gerais (todos os experimentos):")
print(f"  • Alpha - MTI: {avg_mti_alpha:.2f} | STI: {avg_sti_alpha:.2f} | Δ: {avg_mti_alpha - avg_sti_alpha:+.2f}")
print(f"  • Exact % - MTI: {avg_mti_exact:.2f}% | STI: {avg_sti_exact:.2f}% | Δ: {avg_mti_exact - avg_sti_exact:+.2f}%")

print("\n" + "="*100)


🏆 ANÁLISE DE DOMINÂNCIA GLOBAL


NameError: name 'df_diff_overall' is not defined

## 10. Tabelas Detalhadas por Abordagem - Alpha e Exact Match por Métrica

### 10.1 Tabela Detalhada - Abordagem MTI

In [32]:
# Criar tabela detalhada para MTI
mti_table_rows = []

# Definir ordem desejada para os experimentos
experiment_order = ["V1", "V2", "V3", "V4", "V5", "GT"]

for exp in experiment_order:
    row_data = {"Experimento": exp}
    
    # Pegar dados do experimento
    df_exp = df[df["experiment"] == exp]
    df_exp_mti = df_exp[df_exp["approach"] == "MTI"]
    
    # Overall Alpha e Exact Match
    df_overall_mti = df_exp_mti[df_exp_mti["metric"] == "Overall Alpha"]
    if not df_overall_mti.empty:
        overall_alpha = df_overall_mti["krippendorff_alpha"].values[0]
        overall_exact = df_overall_mti["exact_match_percentage"].values[0]
    else:
        overall_alpha = 0
        overall_exact = 0
    
    # Para cada métrica individual
    for metric in METRICS:
        df_metric = df_exp_mti[df_exp_mti["metric"] == metric]
        
        if not df_metric.empty:
            alpha_val = df_metric["krippendorff_alpha"].values[0]
            exact_val = df_metric["exact_match_percentage"].values[0]
            
            # Formatar: "α: 0.12 | 45.67%"
            if pd.notna(exact_val):
                col_value = f"α: {alpha_val:.2f} | {exact_val:.1f}%"
            else:
                col_value = f"α: {alpha_val:.2f}"
        else:
            col_value = "N/A"
        
        # Usar o label da métrica sem quebra de linha
        metric_label = METRIC_LABELS[metric].replace('\n', ' ')
        row_data[metric_label] = col_value
    
    # Adicionar coluna de média geral
    row_data["Média Geral"] = f"α: {overall_alpha:.2f} | {overall_exact:.1f}%"
    
    mti_table_rows.append(row_data)

# Criar DataFrame já ordenado
df_mti_detailed = pd.DataFrame(mti_table_rows)

# Garantir categorização correta e ordenação final (caso o DF real contenha mais itens)
df_mti_detailed["Experimento"] = pd.Categorical(
    df_mti_detailed["Experimento"],
    categories=experiment_order,
    ordered=True
)

df_mti_detailed = df_mti_detailed.sort_values("Experimento")

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM MTI")
print("="*150)
print("Formato: α: [Krippendorff's Alpha] | [Exact Match %]")
print("-"*150)
display(df_mti_detailed)



📊 TABELA DETALHADA - ABORDAGEM MTI
Formato: α: [Krippendorff's Alpha] | [Exact Match %]
------------------------------------------------------------------------------------------------------------------------------------------------------


,Experimento,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média Geral
0,V1,α: -0.43 | 15.4%,α: -0.63 | 17.1%,α: -0.36 | 18.8%,α: -0.23 | 53.3%,α: 0.18 | 60.0%,α: -0.29 | 32.9%
1,V2,α: 0.23 | 65.4%,α: 0.18 | 59.2%,α: 0.19 | 59.2%,α: 0.21 | 94.6%,α: 0.20 | 65.4%,α: 0.20 | 68.8%
2,V3,α: 0.32 | 67.9%,α: 0.03 | 44.6%,α: 0.25 | 62.9%,α: 0.23 | 92.9%,α: 0.12 | 68.8%,α: 0.19 | 67.4%
3,V4,α: 0.28 | 64.2%,α: -0.03 | 42.5%,α: 0.26 | 62.9%,α: 0.29 | 94.6%,α: 0.12 | 69.6%,α: 0.18 | 66.8%
4,V5,α: 0.32 | 65.8%,α: 0.24 | 56.7%,α: 0.23 | 58.3%,α: 0.35 | 94.6%,α: 0.28 | 67.1%,α: 0.28 | 68.5%
5,GT,α: 0.39 | 72.5%,α: 0.39 | 71.7%,α: 0.38 | 71.2%,α: 0.24 | 97.5%,α: 0.40 | 72.9%,α: 0.36 | 77.2%


### 10.2 Tabela Detalhada - Abordagem STI

In [33]:
# Criar tabela detalhada para STI
sti_table_rows = []

# Definir ordem desejada para os experimentos
experiment_order = ["V1", "V2", "V3", "V4", "V5", "GT"]

for exp in experiment_order:
    row_data = {"Experimento": exp}
    
    # Pegar dados do experimento
    df_exp = df[df["experiment"] == exp]
    df_exp_sti = df_exp[df_exp["approach"] == "STI"]
    
    # Overall Alpha e Exact Match
    df_overall_sti = df_exp_sti[df_exp_sti["metric"] == "Overall Alpha"]
    if not df_overall_sti.empty:
        overall_alpha = df_overall_sti["krippendorff_alpha"].values[0]
        overall_exact = df_overall_sti["exact_match_percentage"].values[0]
    else:
        overall_alpha = 0
        overall_exact = 0
    
    # Para cada métrica individual
    for metric in METRICS:
        df_metric = df_exp_sti[df_exp_sti["metric"] == metric]
        
        if not df_metric.empty:
            alpha_val = df_metric["krippendorff_alpha"].values[0]
            exact_val = df_metric["exact_match_percentage"].values[0]
            
            # Formatar: "α: 0.12 | 45.67%"
            if pd.notna(exact_val):
                col_value = f"α: {alpha_val:.2f} | {exact_val:.1f}%"
            else:
                col_value = f"α: {alpha_val:.2f}"
        else:
            col_value = "N/A"
        
        # Usar o label da métrica sem quebra de linha
        metric_label = METRIC_LABELS[metric].replace('\n', ' ')
        row_data[metric_label] = col_value
    
    # Adicionar coluna de média geral
    row_data["Média Geral"] = f"α: {overall_alpha:.2f} | {overall_exact:.2f}%"
    
    sti_table_rows.append(row_data)

df_sti_detailed = pd.DataFrame(sti_table_rows)

# Aplicar categorização para garantir ordenação correta
df_sti_detailed["Experimento"] = pd.Categorical(
    df_sti_detailed["Experimento"],
    categories=experiment_order,
    ordered=True
)

df_sti_detailed = df_sti_detailed.sort_values("Experimento")

print("\n" + "="*150)
print("📊 TABELA DETALHADA - ABORDAGEM STI")
print("="*150)
print("Formato: α: [Krippendorff's Alpha] | [Exact Match %]")
print("-"*150)
display(df_sti_detailed)



📊 TABELA DETALHADA - ABORDAGEM STI
Formato: α: [Krippendorff's Alpha] | [Exact Match %]
------------------------------------------------------------------------------------------------------------------------------------------------------


,Experimento,Coerência,Especificidade,Informatividade,Relevância,Compreensibilidade,Média Geral
0,V1,α: -0.50 | 10.8%,α: -0.59 | 25.0%,α: -0.46 | 12.9%,α: -0.38 | 41.2%,α: 0.15 | 55.8%,α: -0.36 | 29.17%
1,V2,α: 0.14 | 64.6%,α: 0.16 | 62.9%,α: 0.15 | 62.5%,α: 0.40 | 98.8%,α: 0.13 | 64.2%,α: 0.20 | 70.58%
2,V3,α: 0.10 | 62.1%,α: -0.05 | 45.8%,α: 0.07 | 59.2%,α: 0.24 | 97.1%,α: -0.10 | 65.8%,α: 0.05 | 66.00%
3,V4,α: 0.20 | 63.8%,α: -0.02 | 47.9%,α: 0.19 | 62.1%,α: 0.28 | 97.9%,α: 0.01 | 67.9%,α: 0.13 | 67.92%
4,V5,α: 0.14 | 62.9%,α: 0.13 | 58.3%,α: 0.16 | 60.8%,α: 0.19 | 96.2%,α: 0.11 | 63.3%,α: 0.14 | 68.33%
5,GT,α: 0.49 | 75.8%,α: 0.50 | 75.8%,α: 0.49 | 75.4%,α: -0.01 | 97.9%,α: 0.48 | 75.4%,α: 0.39 | 80.08%


## 11. Exportação das Tabelas para LaTeX e PNG

In [36]:
import dataframe_image as dfi
from datetime import datetime

# Criar diretório para exportação se não existir
export_dir = "inference/final_metrics/exports"
os.makedirs(export_dir, exist_ok=True)

# Timestamp para nomes de arquivo únicos
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("🚀 Iniciando exportação das tabelas...")
print("="*100)

🚀 Iniciando exportação das tabelas...


### 11.1 Exportar Tabela MTI

In [37]:
# Exportar Tabela MTI para LaTeX
mti_latex_file = f"{export_dir}/tabela_mti_krippendorff.tex"
mti_png_file = f"{export_dir}/tabela_mti_krippendorff.png"

# Configurar estilo para melhor visualização no LaTeX
df_mti_styled = df_mti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4472C4'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Função para gerar LaTeX customizado no formato do exemplo
def generate_custom_latex(df, caption, label, approach_name):
    """Gera código LaTeX formatado como no exemplo"""
    
    # Cabeçalho do LaTeX
    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append(f"\\caption{{{caption}}}")
    latex_lines.append(f"\\label{{{label}}}")
    latex_lines.append("\\centering")
    latex_lines.append("")
    latex_lines.append("\\begin{adjustbox}{max width=\\textwidth}")
    
    # Determinar número de colunas (Experimento + métricas)
    num_cols = len(df.columns)
    col_format = 'l' + 'c' * (num_cols - 1)
    
    latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
    latex_lines.append("\\toprule")
    
    # Cabeçalho da tabela
    header_parts = []
    for col in df.columns:
        if col == "Experimento":
            header_parts.append("Experimento")
        else:
            header_parts.append(col)
    
    latex_lines.append(" & ".join(header_parts) + " \\\\")
    latex_lines.append("\\midrule")
    
    # Linhas de dados
    for idx, row in df.iterrows():
        row_parts = []
        for col in df.columns:
            cell_value = row[col]
            
            if col == "Experimento":
                # Nome do experimento (limpar prefixo se necessário)
                exp_name = str(cell_value).replace("refine_judge_gpt-4o-mini-2024-07-18_", "")
                exp_name = exp_name.replace("_CORRETO", "")
                row_parts.append(exp_name)
            else:
                # Processar valores de métrica (formato: "α: 0.123 | 45.6%")
                cell_str = str(cell_value)
                
                if "N/A" in cell_str:
                    row_parts.append("N/A")
                else:
                    # Parsear o formato "α: 0.123 | 45.6%"
                    if "|" in cell_str:
                        parts = cell_str.split("|")
                        alpha_part = parts[0].strip()  # "α: 0.123"
                        exact_part = parts[1].strip()  # "45.6%"
                        
                        # Extrair valores numéricos
                        alpha_value = alpha_part.replace("α:", "").strip()
                        exact_value = exact_part.replace("%", "").strip()
                        
                        # Formatar no padrão LaTeX com eM
                        latex_cell = f"$\\alpha$: {alpha_value} | eM: {exact_value}\\%"
                        row_parts.append(latex_cell)
                    else:
                        # Caso só tenha alpha
                        alpha_value = cell_str.replace("α:", "").strip()
                        latex_cell = f"$\\alpha$: {alpha_value}"
                        row_parts.append(latex_cell)
        
        latex_lines.append(" & ".join(row_parts) + " \\\\")
    
    # Rodapé da tabela
    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{adjustbox}")
    latex_lines.append("")
    latex_lines.append("\\end{table}")
    
    return "\n".join(latex_lines)

# Gerar LaTeX customizado para MTI
latex_output = generate_custom_latex(
    df_mti_detailed,
    caption="Resultados Detalhados da Abordagem MTI - Krippendorff's Alpha e Exact Match por Métrica",
    label="tab:mti_krippendorff",
    approach_name="MTI"
)

# Salvar LaTeX
with open(mti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_mti_styled, mti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela MTI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib se dataframe_image não funcionar
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_mti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_mti_detailed.values, 
                     colLabels=df_mti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_mti_detailed.columns)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_mti_detailed) + 1):
        for j in range(len(df_mti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(mti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela MTI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {mti_latex_file}")
print(f"   🖼️  PNG: {mti_png_file}")
print("-"*100)

⚠️ Usando método alternativo para PNG: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
✅ Tabela MTI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/final_metrics/exports/tabela_mti_krippendorff.tex
   🖼️  PNG: inference/final_metrics/exports/tabela_mti_krippendorff.png
----------------------------------------------------------------------------------------------------
✅ Tabela MTI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/final_metrics/exports/tabela_mti_krippendorff.tex
   🖼️  PNG: inference/final_metrics/exports/tabela_mti_krippendorff.png
----------------------------------------------------------------------------------------------------


### 11.2 Exportar Tabela STI

In [38]:
# Exportar Tabela STI para LaTeX
sti_latex_file = f"{export_dir}/tabela_sti_krippendorff.tex"
sti_png_file = f"{export_dir}/tabela_sti_krippendorff.png"

# Configurar estilo para melhor visualização no LaTeX
df_sti_styled = df_sti_detailed.style.set_properties(**{
    'text-align': 'center',
    'font-size': '10pt',
    'border': '1px solid black'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1A76FF'), ('color', 'white'), 
                                   ('font-weight', 'bold'), ('text-align', 'center'),
                                   ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '100%')]}
])

# Gerar LaTeX customizado para STI (usa a mesma função definida anteriormente)
latex_output = generate_custom_latex(
    df_sti_detailed,
    caption="Resultados Detalhados da Abordagem STI - Krippendorff's Alpha e Exact Match por Métrica",
    label="tab:sti_krippendorff",
    approach_name="STI"
)

# Salvar LaTeX
with open(sti_latex_file, 'w', encoding='utf-8') as f:
    f.write(latex_output)

# Exportar para PNG
try:
    dfi.export(df_sti_styled, sti_png_file, max_rows=-1, max_cols=-1)
    print(f"✅ Tabela STI exportada com sucesso!")
except Exception as e:
    # Alternativa usando matplotlib se dataframe_image não funcionar
    print(f"⚠️ Usando método alternativo para PNG: {str(e)}")
    
    fig, ax = plt.subplots(figsize=(20, len(df_sti_detailed) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    
    table = ax.table(cellText=df_sti_detailed.values, 
                     colLabels=df_sti_detailed.columns,
                     cellLoc='center', 
                     loc='center',
                     bbox=[0, 0, 1, 1])
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 2)
    
    # Estilizar cabeçalho
    for i in range(len(df_sti_detailed.columns)):
        table[(0, i)].set_facecolor('#1A76FF')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Estilizar células
    for i in range(1, len(df_sti_detailed) + 1):
        for j in range(len(df_sti_detailed.columns)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    plt.savefig(sti_png_file, dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.close()
    print(f"✅ Tabela STI exportada com sucesso (método alternativo)!")

print(f"   📄 LaTeX: {sti_latex_file}")
print(f"   🖼️  PNG: {sti_png_file}")
print("-"*100)

⚠️ Usando método alternativo para PNG: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.
✅ Tabela STI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/final_metrics/exports/tabela_sti_krippendorff.tex
   🖼️  PNG: inference/final_metrics/exports/tabela_sti_krippendorff.png
----------------------------------------------------------------------------------------------------
✅ Tabela STI exportada com sucesso (método alternativo)!
   📄 LaTeX: inference/final_metrics/exports/tabela_sti_krippendorff.tex
   🖼️  PNG: inference/final_metrics/exports/tabela_sti_krippendorff.png
----------------------------------------------------------------------------------------------------


### 11.3 Resumo da Exportação e Instruções de Uso no LaTeX

In [ ]:
print("\n" + "="*100)
print("📦 RESUMO DA EXPORTAÇÃO")
print("="*100)

print(f"\n✅ Arquivos exportados com sucesso para o diretório: {export_dir}\n")

print("📊 TABELA MTI (Multi-Task Inference):")
print(f"   • Arquivo LaTeX: {mti_latex_file}")
print(f"   • Arquivo PNG: {mti_png_file}")

print("\n📊 TABELA STI (Single-Task Inference):")
print(f"   • Arquivo LaTeX: {sti_latex_file}")
print(f"   • Arquivo PNG: {sti_png_file}")

print("\n" + "="*100)
print("📝 INSTRUÇÕES DE USO NO LATEX")
print("="*100)

print("""
Para usar as tabelas no seu documento LaTeX:

1️⃣ OPÇÃO 1 - Usar arquivo .tex (Recomendado):
   
   No preâmbulo do seu documento, adicione:
   \\usepackage{booktabs}
   \\usepackage{longtable}
   
   No corpo do documento, onde deseja inserir a tabela:
   \\input{path/to/tabela_mti_krippendorff.tex}
   \\input{path/to/tabela_sti_krippendorff.tex}

2️⃣ OPÇÃO 2 - Usar imagem PNG:
   
   No preâmbulo:
   \\usepackage{graphicx}
   
   No corpo do documento:
   \\begin{figure}[htbp]
       \\centering
       \\includegraphics[width=\\textwidth]{path/to/tabela_mti_krippendorff.png}
       \\caption{Resultados MTI - Krippendorff's Alpha e Exact Match}
       \\label{fig:mti_results}
   \\end{figure}

3️⃣ DICAS:
   • As tabelas já incluem caption e label
   • Para tabelas em formato paisagem, use: \\usepackage{rotating} e \\begin{sidewaystable}
   • Para ajustar o tamanho da fonte: \\small ou \\footnotesize antes da tabela
   • As referências às tabelas podem ser feitas com \\ref{tab:mti_krippendorff}
""")

print("="*100)
print(f"🎉 Exportação concluída com sucesso!")
print("="*100)

In [1]:
# refine_judge_similarity_k_alpha document example:

# {
#   "_id": {
#     "$oid": "692616fed09fd9fa4034f6f1"
#   },
#   "experiment_name": "refine_judge_gpt-4o-mini-2024-07-18_V4_CORRETO",
#   "prompt": "\n[System]\nAct as a neutral and unbiased evaluator. You will evaluate two AI-generated answers to the same user query. For each answer, follow this evaluation procedure:\n\n1. Before choosing a score, briefly outline your reasoning process and then summarize it in a short (1–2 sentence) explanation for each score.\n2. Assign a score from 1 to 5 following the Likert scale for each attribute:\n   - Coherence\n   - Specificity\n   - Informativeness\n   - Relevance\n   - Understandability\n\nFairness constraints:\n- Avoid any position bias: the order in which the answers appear must not influence your evaluation.\n- Do not allow response length to affect your judgment.\n- Do not favor any assistant based on its name or label.\n\n---\nLikert Scale Definitions (1–5):\n#### 1 — Very Poor\n- Coherence: The response is disorganized, contradictory, or lacks logical flow.  \n- Specificity: Extremely vague; provides generic statements unrelated to the query.  \n- Informativeness: Adds little to no meaningful content; omits essential information.  \n- Relevance: Largely off-topic or addresses only a small fraction of the intended task.  \n- Understandability: The answer is very hard to follow: sentences are confusing, grammar or structure severely obstruct meaning, and the reader cannot reliably extract the intended message.\n\n#### 2 — Poor\n- Coherence: Some isolated logical elements exist, but major gaps hinder understanding.  \n- Specificity: Mostly generic; few details are present and they do not add much value.  \n- Informativeness: Limited content; misses several key aspects expected in a good answer.  \n- Relevance: Partially related but includes irrelevant or misplaced sections.  \n- Understandability: The response can be understood in parts but contains ambiguous phrasing, grammatical issues, or awkward structure that require effort to interpret and may lead to misunderstanding.\n\n#### 3 — Fair\n- Coherence: Generally logical but may have jumps, weak transitions, or mild inconsistencies.  \n- Specificity: Includes a mix of general and task-specific elements; adequate but not strong.  \n- Informativeness: Covers important points but may miss nuances or depth.  \n- Relevance: Mostly stays on topic with occasional unnecessary or unfocused content.  \n- Understandability: Readable and mostly clear; some sentences or terms are imprecise or slightly confusing, but the overall meaning is recoverable without excessive effort.\n\n#### 4 — Good\n- Coherence: Well-structured and easy to follow, with clear logical connections.  \n- Specificity: Provides meaningful and relevant details tailored to the query.  \n- Informativeness: Delivers substantial and accurate information; minor gaps may exist.  \n- Relevance: Strongly aligned with the task; minimal drift or redundancy.  \n- Understandability: The answer is clearly expressed with appropriate sentence structure and vocabulary; minor phrasing issues may appear but do not hamper comprehension.\n\n#### 5 — Excellent\n- Coherence: Highly organized, internally consistent, and logically seamless.  \n- Specificity: Rich in precise, context-specific details without unnecessary generalities.  \n- Informativeness: Comprehensive, insightful, and fully addresses all key aspects.  \n- Relevance: Perfectly aligned with the question, with zero irrelevant content.  \n- Understandability: Exceptionally clear and easy to read: grammar and syntax are correct, terminology is used precisely, sentences are well-formed, and a reader can immediately grasp the intended meaning without ambiguity.\n---\n\nFor each answer, output:\n- A JSON object (scores_a / scores_b) containing the numerical scores.\n- A JSON object (explanations_a / explanations_b) containing the explanations.\n- Output only the JSON objects as plain text, with no extra formatting.\n\nYour output should follow exactly this template:\nscores_a = { \"Coherence\": 4, \"Specific\": 3, \"Informativeness\": 4, \"Relevance\": 3, \"Understandability\": 5 }  \nexplanations_a = {\n    \"Coherence\": \"The response is well-structured and follows a clear logical flow, with only minor issues in transitions or organization.\",\n    \"Specificity\": \"The content provides adequate task-related details but still mixes general and specific elements.\",\n    \"Informativeness\": \"The response delivers solid and useful information, covering the main points well, though it may miss some finer nuances.\",\n    \"Relevance\": \"The response stays mostly on topic, with only occasional unnecessary or unfocused content that slightly reduces precision.\",\n    \"Understandability\": \"The response is exceptionally clear and easy to read; its structure and language allow immediate comprehension without ambiguity.\"\n}\nscores_b = { ... }\nexplanations_b = { ... }\n\n[User Question]\n<|user|> ### Example:\n\n### Instruction: Read the following passage, and follow the given steps.\n#1: Transate the given text to German.\n#2: Based on the text you have translated in step#1 and solve the question. Return the answer in <task2>N<task2/> format.\n\n###Text\nTaiwan beginning start way back in 15th century where European sailors passing by record the island’s name as Ilha Formosa, or beautiful island. In 1624,Dutch East India Company establishes a base in southwestern Taiwan, initiating a transformation in aboriginal grain production practices and employing Chinese laborers to work on its rice and sugar plantations. In 1683, Qing dynasty (1644-1912) forces take control of Taiwan’s western and northern coastal areas and declared Taiwan as a province of the Qing Empire in 1885. In 1895, after defeat in the First Sino-Japanese War (1894-1895), the Qing government signs the Treaty of Shimonoseki, by which it cedes sovereignty over Taiwan to Japan, which rules the island until 1945.\n###Question:\nWas trifft nicht auf die Niederländische Ostindien-Kompanie während ihrer Zeit in Taiwan zu?\n###Options:\n1. Sie hatten eine Niederlassung im Südwesten der Insel\n2. Sie praktizierten die Getreideanbaumethoden der Ureinwohner\n3. Sie beschäftigten chinesische Arbeiter auf ihren Plantagen\n4. Sie traten die Souveränität über Taiwan an die Qing-Dynastie ab\n\n###Answer:\n\n###Instruction1:\nDer Beginn Taiwans geht auf das 15. Jahrhundert zurück, als europäische Seefahrer dort vorbeifuhren und den Namen der Insel als Ilha Formosa bzw. schöne Insel dokumentierten. 1624 gründet die Niederländische Ostindien-Kompanie eine Niederlassung im Südwesten Taiwans, leitet damit einen Wandel in den Getreideanbaumethoden der Ureinwohner ein und beschäftigt chinesische Arbeiter auf ihren Reis- und Zuckerplantagen. Im Jahr 1683 übernehmen Streitkräfte der Qing-Dynastie (1644-1912) die Kontrolle über Taiwans westliche und nördliche Küstenregionen und erklärten im Jahr 1885 Taiwan zur Provinz des Qing-Reiches. 1895 unterzeichnete die Regierung der Qing-Dynastie nach der Niederlage im ersten Chinesisch-Japanischen Krieg (1894-1895) den Vertrag von Shimonoseki, in dem sie die Souveränität über Taiwan an Japan abtrat, welches die Insel bis 1945 regierte.\n\n###Instruction2:\nDie Niederländische Ostindien-Kompanie hat nicht die Souveränität über Taiwan an die Qing-Dynastie abgetreten. Diese Information ist im Text nicht enthalten. Daher ist Aussage 4 falsch.\n\n<task2>4<task2/>\n\n### Task:\n\n### Instruction: Read the following passage, and follow the given steps.\n#1: Transate the given text to German.\n#2: Based on the text you have translated in step#1 and solve the question. Return the answer in <task2>N<task2/> format.\n\n###Text\nBird flu, or more formally avian influenza, can infect both birds and mammals. Fewer than a thousand cases have ever been reported in humans, but some of them have been fatal. Most have involved people who work with poultry, but there is also some risk to birdwatchers.\n###Question:\nWelche der folgenden Aussagen trifft auf die Vogelgrippe dem Abschnitt nach zu?\n###Options:\n1. Nur Menschen, die mit Geflügel arbeiten, haben sich mit der Grippe angesteckt\n2. Beim Menschen gibt es mehr dokumentierte Fälle als bei Vögeln\n3. Für Vogelbeobachter besteht ein Risiko, sich anzustecken\n4. Alle beim Menschen dokumentierten Fälle waren tödlich\n\n### Answer:\n<|assistant|>\n\n[Assistant A Response]\n### Instruction1:\nVogelgrippe, oder formell gesagt avian Influenza, kann sowohl Vögel als auch Säugetiere infizieren. Es wurden bisher weniger als tausend Fälle beim Menschen gemeldet, aber einige davon waren tödlich. Die meisten betrafen Menschen, die mit Geflügel arbeiten, aber es besteht auch ein gewisses Risiko für Vogelbeobachter.\n\n### Instruction2:\nDie Aussage, dass ein Risiko für Vogelbeobachter besteht, ist im Text enthalten. Die anderen Aussagen sind entweder falsch oder nicht enthalten. Daher ist Aussage 3 korrekt.\n\n<task2>3<task2/>\n\n[Assistant B Response]\n### Instruction1:\n\nVogelgrippe, oder formeller als aviäre Influenza bezeichnet, kann sowohl Vögel als auch Säugetiere infizieren. Weniger als tausend Fälle wurden jemals bei Menschen gemeldet, aber einige davon waren tödlich. Die meisten betrafen Personen, die mit Geflügel arbeiten, aber es gibt auch ein gewisses Risiko für Vogelbeobachter.\n\n### Instruction2:\n\nDie Frage lautet: \"Welche der folgenden Aussagen trifft auf die Vogelgrippe dem Abschnitt nach zu?\". Laut dem übersetzten Text besteht ein gewisses Risiko für Vogelbeobachter, sich mit der Vogelgrippe anzustecken. Daher lautet die Antwort auf die Frage:\n\n<task2>3<task2/>\n\n",
#   "sti_k_alpha": 0.13078323338862288,
#   "mti_k_alpha": 0.18459000470402964,
#   "sti_k_alpha_by_metric": {
#     "coherence": 0.20032287617624023,
#     "specificity": -0.019844062530433693,
#     "informativeness": 0.19171746514025867,
#     "relevance": 0.2766535789791603,
#     "Understandability": 0.005066309177888861
#   },
#   "mti_k_alpha_by_metric": {
#     "coherence": 0.28392387799469243,
#     "specificity": -0.03430404231305939,
#     "informativeness": 0.2591895911411909,
#     "relevance": 0.28907409521634897,
#     "Understandability": 0.1250665014809753
#   },
#   "sti_exact_match_percentage": 67.91666666666666,
#   "mti_exact_match_percentage": 66.75,
#   "sti_exact_match_percentage_by_metric": {
#     "coherence": 63.74999999999999,
#     "specificity": 47.91666666666667,
#     "informativeness": 62.083333333333336,
#     "relevance": 97.91666666666666,
#     "Understandability": 67.91666666666667
#   },
#   "mti_exact_match_percentage_by_metric": {
#     "coherence": 64.16666666666667,
#     "specificity": 42.5,
#     "informativeness": 62.916666666666664,
#     "relevance": 94.58333333333333,
#     "Understandability": 69.58333333333333
#   },
#   "calculation_timestamp": {
#     "$date": "2025-11-25T17:52:14.490Z"
#   },
#   "matched_documents": 240,
#   "total_documents": 240
# }

In [ ]:
"LateX code format example":

"""
\begin{table}[htbp]
\caption{Resultados Detalhados da Abordagem MTI - Krippendorff's Alpha e Exact Match por Métrica}
\label{tab:mti_krippendorff}
\centering

\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Experimento & Coerência & Especificidade & Informatividade & Relevância & Compreensibilidade & Média Geral \\
\midrule
V1 & $\alpha$: -0.43 | 15.4\% & $\alpha$: -0.63 | 17.1\% & $\alpha$: -0.36 | 18.8\% & $\alpha$: -0.23 | 53.3\% & $\alpha$: 0.18 | 60.0\% & $\alpha$: -0.30 | 32.9\% \\
V2 & $\alpha$: 0.23 | 65.4\% & $\alpha$: 0.18 | 59.2\% & $\alpha$: 0.19 | 59.2\% & $\alpha$: 0.21 | 94.6\% & $\alpha$: 0.20 | 65.4\% & $\alpha$: 0.20 | 68.8\% \\
V3 & $\alpha$: 0.32 | 67.9\% & $\alpha$: 0.03 | 44.6\% & $\alpha$: 0.25 | 62.9\% & $\alpha$: 0.23 | 92.9\% & $\alpha$: 0.12 | 68.8\% & $\alpha$: 0.19 | 67.4\% \\
V4 & $\alpha$: 0.28 | 64.2\% & $\alpha$: -0.03 | 42.5\% & $\alpha$: 0.26 | 62.9\% & $\alpha$: 0.29 | 94.6\% & $\alpha$: 0.13 | 69.6\% & $\alpha$: 0.19 | 66.8\% \\
V5 & $\alpha$: 0.32 | 65.8\% & $\alpha$: 0.25 | 56.7\% & $\alpha$: 0.23 | 58.3\% & $\alpha$: 0.35 | 94.6\% & $\alpha$: 0.28 | 67.1\% & $\alpha$: 0.29 | 68.5\% \\
GT & $\alpha$: 0.32 | 67.9\% & $\alpha$: 0.03 | 44.6\% & $\alpha$: 0.25 | 62.9\% & $\alpha$: 0.23 | 92.9\% & $\alpha$: 0.12 | 68.8\% & $\alpha$: 0.19 | 67.4\% \\
\bottomrule
\end{tabular}
\end{adjustbox}

\end{table}
"""